In [1]:
# Run this cell first to install all required packages
import subprocess
import sys

packages = [
    'selenium',
    'webdriver-manager',
    'requests',
    'beautifulsoup4',
    'pandas',
    'numpy',
    'xgboost',
    'scikit-learn',
    'joblib',
    'fake-useragent',
    'openpyxl',
    'matplotlib',
    'tqdm'
]

for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
    print(f"✅ Installed: {package}")

print("\n✅ All packages installed successfully!")

✅ Installed: selenium
✅ Installed: webdriver-manager
✅ Installed: requests
✅ Installed: beautifulsoup4
✅ Installed: pandas
✅ Installed: numpy
✅ Installed: xgboost
✅ Installed: scikit-learn
✅ Installed: joblib
✅ Installed: fake-useragent
✅ Installed: openpyxl
✅ Installed: matplotlib
✅ Installed: tqdm

✅ All packages installed successfully!


In [2]:
import os
import re
import json
import time
import random
import logging
import warnings
import pickle
from typing import Optional, Dict, List, Tuple
from urllib.parse import urlparse, urljoin
from datetime import datetime
from pathlib import Path

# Web scraping
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.common.exceptions import TimeoutException, NoSuchElementException, WebDriverException

# For Chrome driver management
from webdriver_manager.chrome import ChromeDriverManager
import chromedriver_autoinstaller

# Data handling
import pandas as pd
import numpy as np

# ML imports
import joblib
import xgboost as xgb
from sklearn.preprocessing import StandardScaler

# Visualization (optional)
import matplotlib.pyplot as plt
from IPython.display import display, HTML

# Progress bar
from tqdm import tqdm

# Suppress warnings
warnings.filterwarnings('ignore')

print("✅ All imports successful!")
print(f"📁 Current working directory: {os.getcwd()}")

✅ All imports successful!
📁 Current working directory: c:\Users\disha\OneDrive\Documents\company_website_predictor


In [4]:
# ==================== INPUT YOUR DATA HERE ====================

# 1️⃣ INPUT: Your Serper API Key (get from https://serper.dev/)
SERPER_API_KEY = "efa6cbe513613e548d0742a7e5cf6cab1ea82a01"  # <-- CHANGE THIS

# 2️⃣ INPUT: Your XGBoost Model Path
MODEL_PATH = "company_predictor_XGBmodel.joblib"  # <-- CHANGE THIS

# 3️⃣ INPUT: Load companies from CSV
csv_file = 'test_batch2.csv'  # <-- CHANGE THIS

# ==================== VERIFY MODEL ====================
print("🔍 Checking model file...")
if os.path.exists(MODEL_PATH):
    file_size = os.path.getsize(MODEL_PATH) / (1024 * 1024)
    print(f"✅ Model file found: {MODEL_PATH}")
    print(f"📦 File size: {file_size:.2f} MB")
    
    try:
        xgb_model = joblib.load(MODEL_PATH)
        print(f"✅ Model loaded successfully! Type: {type(xgb_model)}")
        
        if hasattr(xgb_model, 'n_features_in_'):
            print(f"📈 Expected features: {xgb_model.n_features_in_}")
    except Exception as e:
        print(f"❌ Error loading model: {e}")
        raise
else:
    print(f"❌ Model file not found: {MODEL_PATH}")
    print("📂 Files in current directory:")
    for file in os.listdir('.'):
        if file.endswith(('.joblib', '.pkl', '.pickle')):
            print(f"  - {file} (model file found!)")
    raise FileNotFoundError(f"Model file {MODEL_PATH} not found")

# ==================== LOAD COMPANIES ====================
if os.path.exists(csv_file):
    df = pd.read_csv(csv_file)
    print(f"\n📋 CSV Columns: {df.columns.tolist()}")
    print(f"📊 First 3 rows:")
    print(df.head(3))
    
    # IMPORTANT: Change 'company_name' to match your CSV column name
    # If your column is 'Company Name', use df['Company Name'].tolist()
    # If your column is 'company', use df['company'].tolist()
    # If your column is 'Name', use df['Name'].tolist()
    COMPANY_NAMES = df['Company Name'].tolist()  # <-- CHANGE COLUMN NAME
    
    print(f"\n✅ Loaded {len(COMPANY_NAMES)} companies from {csv_file}")
    print(f"📊 First 5 companies: {COMPANY_NAMES[:5]}")
else:
    print(f"❌ CSV file '{csv_file}' not found!")
    print("📂 CSV files in current directory:")
    csv_files = [f for f in os.listdir('.') if f.endswith('.csv')]
    if csv_files:
        for f in csv_files:
            print(f"  - {f}")
    else:
        print("  No CSV files found!")
    
    # Fallback hardcoded list
    COMPANY_NAMES = [
        "Apple Inc.",
        "Microsoft Corporation", 
        "Google LLC",
        "Tesla Inc.",
        "Netflix Inc."
    ]
    print(f"\n⚠️  Using fallback list with {len(COMPANY_NAMES)} companies")

print("\n" + "="*50)
print("✅ All inputs ready!")
print(f"📊 Companies to process: {len(COMPANY_NAMES)}")
print(f"🔧 Model: {MODEL_PATH}")
print(f"📁 CSV file: {csv_file}")
print("="*50)

🔍 Checking model file...
✅ Model file found: company_predictor_XGBmodel.joblib
📦 File size: 0.13 MB
✅ Model loaded successfully! Type: <class 'xgboost.sklearn.XGBClassifier'>
📈 Expected features: 4

📋 CSV Columns: ['Company Name']
📊 First 3 rows:
            Company Name
0     Berkshire Hathaway
1  General Electric (GE)
2     Oracle Corporation

✅ Loaded 20 companies from test_batch2.csv
📊 First 5 companies: ['Berkshire Hathaway', 'General Electric (GE)', 'Oracle Corporation', 'Adobe Inc.', 'Cisco Systems']

✅ All inputs ready!
📊 Companies to process: 20
🔧 Model: company_predictor_XGBmodel.joblib
📁 CSV file: test_batch2.csv


In [4]:
# ==================== CELL 4: LINKEDIN SCRAPER (IMPROVED) ====================
import time
import random
import re
import json
from typing import Optional, Dict, List
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

class LinkedInScraper:
    """Robust LinkedIn scraper with multiple extraction methods"""
    
    def __init__(self, headless: bool = True):
        self.headless = headless
        self.driver = None
        self.setup_logging()
        
    def setup_logging(self):
        logging.basicConfig(
            level=logging.WARNING,
            format='%(asctime)s - %(levelname)s - %(message)s'
        )
        self.logger = logging.getLogger(__name__)
    
    def create_driver(self) -> webdriver.Chrome:
        """Create Chrome driver with optimal anti-detection settings"""
        chromedriver_autoinstaller.install()
        
        chrome_options = Options()
        
        if self.headless:
            chrome_options.add_argument('--headless=new')
        
        # Anti-detection settings
        chrome_options.add_argument('--disable-blink-features=AutomationControlled')
        chrome_options.add_argument('--disable-dev-shm-usage')
        chrome_options.add_argument('--no-sandbox')
        chrome_options.add_argument('--disable-web-security')
        chrome_options.add_argument('--disable-gpu')
        chrome_options.add_argument('--disable-extensions')
        chrome_options.add_argument('--disable-popup-blocking')
        chrome_options.add_argument('--disable-notifications')
        chrome_options.add_argument('--disable-infobars')
        chrome_options.add_argument('--ignore-certificate-errors')
        chrome_options.add_argument('--window-size=1200,800')
        
        # Random user agent
        user_agents = [
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36',
            'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
        ]
        chrome_options.add_argument(f'user-agent={random.choice(user_agents)}')
        
        chrome_options.add_experimental_option('excludeSwitches', ['enable-automation'])
        chrome_options.add_experimental_option('useAutomationExtension', False)
        chrome_options.add_experimental_option('prefs', {
            'profile.default_content_setting_values.notifications': 2,
            'credentials_enable_service': False,
            'profile.password_manager_enabled': False
        })
        
        service = Service(ChromeDriverManager().install())
        driver = webdriver.Chrome(service=service, options=chrome_options)
        
        driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {
            'source': '''
                Object.defineProperty(navigator, 'webdriver', {
                    get: () => undefined
                });
                window.chrome = {
                    runtime: {}
                };
            '''
        })
        
        return driver
    
    def scrape_linkedin_profile(self, profile_url: str) -> Dict[str, Optional[str]]:
        """Main scraping method with multiple fallback strategies"""
        result = {
            'website': None,
            'company_name': None,
            'description': None,
            'industry': None,
            'employee_count': None,
            'headquarters': None
        }
        
        self.driver = self.create_driver()
        
        try:
            self.logger.info(f"Scraping LinkedIn profile: {profile_url}")
            
            self.driver.get(profile_url)
            time.sleep(random.uniform(3, 5))
            
            # Wait for page to load
            WebDriverWait(self.driver, 15).until(
                EC.presence_of_element_located((By.TAG_NAME, "body"))
            )
            
            # Scroll to load dynamic content
            self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)
            
            # Extract website using multiple methods
            website = self._extract_website_from_button()
            if website and website != 'static.licdn.com':
                result['website'] = website
                self.logger.info(f"Found website via button: {website}")
            
            if not result['website'] or result['website'] == 'static.licdn.com':
                website = self._extract_website_from_text()
                if website and website != 'static.licdn.com':
                    result['website'] = website
                    self.logger.info(f"Found website from text: {website}")
            
            if not result['website'] or result['website'] == 'static.licdn.com':
                website = self._extract_website_from_description()
                if website and website != 'static.licdn.com':
                    result['website'] = website
                    self.logger.info(f"Found website from description: {website}")
            
            if not result['website'] or result['website'] == 'static.licdn.com':
                website = self._extract_website_from_page_source()
                if website and website != 'static.licdn.com':
                    result['website'] = website
                    self.logger.info(f"Found website from page source: {website}")
            
            # Extract other info
            result['company_name'] = self._extract_company_name(profile_url)
            result['description'] = self._extract_description()
            result['industry'] = self._extract_industry()
            
            # Final check - if website is static.licdn.com, set to None
            if result['website'] == 'static.licdn.com':
                result['website'] = None
            
            return result
            
        except Exception as e:
            self.logger.error(f"Error scraping profile: {e}")
            return result
        finally:
            self.cleanup_driver()
    
    def _extract_website_from_button(self) -> Optional[str]:
        """Extract website from the website button/link"""
        try:
            # Look for the website button
            selectors = [
                '.org-page-details__website-link',
                '.link-without-visited-state',
                '.org-top-card__website-link',
                'a[data-anonymize="company-website"]',
                'a[href*="linkedin.com/redir/"]'
            ]
            
            for selector in selectors:
                try:
                    elements = self.driver.find_elements(By.CSS_SELECTOR, selector)
                    for element in elements:
                        href = element.get_attribute('href')
                        text = element.text.strip().lower()
                        
                        if href:
                            # Check if it's a website link
                            if 'website' in text or 'site' in text or 'www.' in href:
                                website = self._clean_website_url(href)
                                if website and website != 'static.licdn.com':
                                    return website
                except:
                    continue
            
            # Try finding by link text
            try:
                links = self.driver.find_elements(By.TAG_NAME, "a")
                for link in links:
                    text = link.text.strip().lower()
                    if 'website' in text or 'visit' in text and 'site' in text:
                        href = link.get_attribute('href')
                        if href:
                            website = self._clean_website_url(href)
                            if website and website != 'static.licdn.com':
                                return website
            except:
                pass
            
            return None
            
        except Exception as e:
            self.logger.error(f"Error extracting website from button: {e}")
            return None
    
    def _extract_website_from_text(self) -> Optional[str]:
        """Extract website from visible text on the page"""
        try:
            # Look for website in the company info section
            page_text = self.driver.find_element(By.TAG_NAME, "body").text
            
            # Look for website patterns
            patterns = [
                r'Website[:\s]*([a-zA-Z0-9-]+\.[a-zA-Z]{2,}(?:\.[a-zA-Z]{2,})?)',
                r'Visit our website[:\s]*([a-zA-Z0-9-]+\.[a-zA-Z]{2,}(?:\.[a-zA-Z]{2,})?)',
                r'Company Website[:\s]*([a-zA-Z0-9-]+\.[a-zA-Z]{2,}(?:\.[a-zA-Z]{2,})?)',
            ]
            
            for pattern in patterns:
                matches = re.findall(pattern, page_text, re.IGNORECASE)
                for match in matches:
                    website = self._clean_website_url(match)
                    if website and website != 'static.licdn.com':
                        return website
            
            return None
            
        except Exception as e:
            self.logger.error(f"Error extracting website from text: {e}")
            return None
    
    def _extract_website_from_description(self) -> Optional[str]:
        """Extract website from company description"""
        try:
            description = self._extract_description()
            if description:
                # Look for URL patterns in description
                url_pattern = r'(?:https?://)?(?:www\.)?([a-zA-Z0-9-]+\.[a-zA-Z]{2,}(?:\.[a-zA-Z]{2,})?)'
                matches = re.findall(url_pattern, description)
                for match in matches:
                    website = self._clean_website_url(match)
                    if website and website != 'static.licdn.com':
                        return website
            
            return None
            
        except Exception as e:
            self.logger.error(f"Error extracting website from description: {e}")
            return None
    
    def _extract_website_from_page_source(self) -> Optional[str]:
        """Extract website from page source"""
        try:
            page_source = self.driver.page_source
            soup = BeautifulSoup(page_source, 'html.parser')
            
            # Look for website in meta tags
            meta_tags = soup.find_all('meta')
            for meta in meta_tags:
                if meta.get('name') and 'website' in meta.get('name', '').lower():
                    content = meta.get('content', '')
                    if content:
                        website = self._clean_website_url(content)
                        if website and website != 'static.licdn.com':
                            return website
            
            # Look for JSON-LD
            script_tags = soup.find_all('script', {'type': 'application/ld+json'})
            for script in script_tags:
                try:
                    data = json.loads(script.string)
                    if isinstance(data, dict):
                        if 'url' in data:
                            website = self._clean_website_url(data['url'])
                            if website and website != 'static.licdn.com':
                                return website
                        if 'sameAs' in data:
                            for url in data['sameAs']:
                                website = self._clean_website_url(url)
                                if website and website != 'static.licdn.com':
                                    return website
                except:
                    continue
            
            return None
            
        except Exception as e:
            self.logger.error(f"Error extracting website from page source: {e}")
            return None
    
    def _extract_company_name(self, profile_url: str) -> Optional[str]:
        """Extract company name"""
        try:
            if 'linkedin.com/company/' in profile_url:
                name = profile_url.split('linkedin.com/company/')[-1].split('/')[0]
                name = name.replace('-', ' ').title()
                return name
            
            title = self.driver.title
            if ' | LinkedIn' in title:
                return title.replace(' | LinkedIn', '').strip()
            
            return None
        except:
            return None
    
    def _extract_description(self) -> Optional[str]:
        """Extract company description"""
        try:
            selectors = [
                '.org-page-details__description-text',
                '.break-words',
                '.org-top-card__description'
            ]
            
            for selector in selectors:
                try:
                    element = self.driver.find_element(By.CSS_SELECTOR, selector)
                    description = element.text.strip()
                    if description:
                        return description
                except:
                    continue
            
            return None
        except:
            return None
    
    def _extract_industry(self) -> Optional[str]:
        """Extract company industry"""
        try:
            selectors = ['.org-page-details__industry', '.org-top-card__industry']
            for selector in selectors:
                try:
                    element = self.driver.find_element(By.CSS_SELECTOR, selector)
                    return element.text.strip()
                except:
                    continue
            return None
        except:
            return None
    
    def _clean_website_url(self, url: str) -> Optional[str]:
        """Clean and validate website URL"""
        if not url:
            return None
        
        try:
            # Handle LinkedIn redirect URLs
            if 'linkedin.com/redir/' in url:
                redirect_match = re.search(r'url=([^&]+)', url)
                if redirect_match:
                    url = redirect_match.group(1)
            
            # Decode URL if needed
            if '%' in url:
                try:
                    url = requests.utils.unquote(url)
                except:
                    pass
            
            # Parse URL
            parsed = urlparse(url)
            
            # If no scheme, add http://
            if not parsed.scheme:
                url = f"http://{url}"
                parsed = urlparse(url)
            
            # Extract domain
            domain = parsed.netloc
            
            # Remove www prefix
            if domain.startswith('www.'):
                domain = domain[4:]
            
            # Remove trailing slashes
            domain = domain.rstrip('/')
            
            # Validate domain
            if self._is_valid_company_website(domain):
                return domain
            
            return None
            
        except Exception as e:
            self.logger.error(f"Error cleaning URL {url}: {e}")
            return None
    
    def _is_valid_company_website(self, domain: str) -> bool:
        """Validate if domain is likely a legitimate company website"""
        if not domain:
            return False
        
        domain_lower = domain.lower()
        
        # Reject common non-company domains
        blacklist = [
            'linkedin.com', 'facebook.com', 'twitter.com', 'instagram.com',
            'youtube.com', 'wikipedia.org', 'glassdoor.com', 'indeed.com',
            'crunchbase.com', 'bloomberg.com', 'reuters.com', 'businesswire.com',
            'prnewswire.com', 'marketwatch.com', 'forbes.com', 'cnbc.com',
            'static.licdn.com'  # <-- Add this to blacklist
        ]
        
        for blocked in blacklist:
            if blocked in domain_lower:
                return False
        
        # Must have at least one dot
        if '.' not in domain:
            return False
        
        # Not an IP address
        if re.match(r'^\d+\.\d+\.\d+\.\d+$', domain):
            return False
        
        # Must end with valid TLD
        valid_tlds = ['.com', '.org', '.net', '.io', '.co', '.uk', '.de', '.fr', '.cn', '.jp', '.in', '.ai']
        if not any(domain_lower.endswith(tld) for tld in valid_tlds):
            return False
        
        return True
    
    def cleanup_driver(self):
        """Clean up WebDriver"""
        if self.driver:
            try:
                self.driver.quit()
            except:
                pass
            self.driver = None
    
    def scrape_with_retry(self, profile_url: str, max_retries: int = 3) -> Dict[str, Optional[str]]:
        """Scrape with automatic retry"""
        for attempt in range(max_retries):
            try:
                result = self.scrape_linkedin_profile(profile_url)
                if result['website'] and result['website'] != 'static.licdn.com':
                    return result
                else:
                    print(f"   Attempt {attempt + 1} failed to find website")
                    if attempt < max_retries - 1:
                        wait_time = random.uniform(5, 10)
                        print(f"   Waiting {wait_time:.1f} seconds before retry...")
                        time.sleep(wait_time)
            except Exception as e:
                print(f"   Attempt {attempt + 1} error: {e}")
                if attempt < max_retries - 1:
                    time.sleep(random.uniform(5, 10))
        
        return {'website': None, 'company_name': None, 'description': None}

print("✅ LinkedIn Scraper loaded successfully!")

✅ LinkedIn Scraper loaded successfully!


In [5]:
# ==================== CELL 5: DOMAIN RESOLVER CLASS (FIXED) ====================
from tqdm import tqdm
import numpy as np

class CompanyDomainResolver:
    """Main pipeline for resolving company names to domains"""
    
    def __init__(self, serper_api_key: str, xgboost_model):
        self.serper_api_key = serper_api_key
        self.xgboost_model = xgboost_model
        self.linkedin_scraper = LinkedInScraper(headless=True)
        self.setup_logging()
    
    def setup_logging(self):
        logging.basicConfig(level=logging.WARNING)
        self.logger = logging.getLogger(__name__)
    
    def get_linkedin_profile_url(self, company_name: str) -> Optional[str]:
        """Get LinkedIn profile URL using Serper API"""
        try:
            search_query = f"{company_name} LinkedIn"
            
            params = {
                'api_key': self.serper_api_key,
                'q': search_query,
                'num': 5
            }
            
            response = requests.get('https://google.serper.dev/search', params=params)
            response.raise_for_status()
            data = response.json()
            
            for result in data.get('organic', []):
                link = result.get('link', '')
                if 'linkedin.com/company/' in link:
                    return link
            
            return None
            
        except Exception as e:
            self.logger.error(f"Error fetching LinkedIn URL for {company_name}: {e}")
            return None
    
    def resolve_domain(self, company_name: str) -> Tuple[Optional[str], str]:
        """Resolve company name to domain"""
        # Tier 1: LinkedIn
        self.logger.info(f"Tier 1: Trying LinkedIn for {company_name}")
        
        try:
            linkedin_url = self.get_linkedin_profile_url(company_name)
            
            if linkedin_url:
                result = self.linkedin_scraper.scrape_with_retry(linkedin_url)
                website = result.get('website')
                
                if website and website != 'static.licdn.com':
                    self.logger.info(f"Tier 1 success: {company_name} -> {website}")
                    return (website, 'LinkedIn')
            
        except Exception as e:
            self.logger.error(f"Tier 1 error for {company_name}: {e}")
        
        # Tier 2: ML Fallback
        self.logger.info(f"Tier 2: ML Fallback for {company_name}")
        
        try:
            # Get search results
            search_results = self.get_google_search_results(company_name)
            
            if search_results and len(search_results) >= 3:
                # Prepare features for each of the top 3 results
                best_domain = None
                best_score = -1
                
                for idx in range(3):
                    result = search_results[idx]
                    features = self.prepare_ml_features_for_result(company_name, result, idx)
                    
                    if features is not None:
                        features_array = np.array(features).reshape(1, -1)
                        prediction = self.xgboost_model.predict(features_array)[0]
                        
                        # If prediction is 1, this result is good
                        if prediction == 1:
                            link = result.get('link', '')
                            domain = urlparse(link).netloc
                            if domain.startswith('www.'):
                                domain = domain[4:]
                            
                            if domain and self._is_valid_company_domain(domain):
                                best_domain = domain
                                break
                
                if best_domain:
                    self.logger.info(f"Tier 2 success: {company_name} -> {best_domain}")
                    return (best_domain, 'ML_Fallback')
        
        except Exception as e:
            self.logger.error(f"Tier 2 error for {company_name}: {e}")
        
        return (None, 'Failed')
    
    def prepare_ml_features_for_result(self, company_name: str, result: Dict, position: int) -> Optional[List[float]]:
        """Prepare features for a specific search result"""
        try:
            link = result.get('link', '')
            
            # Clean domain
            domain = urlparse(link).netloc
            if domain.startswith('www.'):
                domain = domain[4:]
            
            # Feature 1: search_position
            position_feature = position + 1
            
            # Feature 2: clean_domain_similarity
            clean_name = re.sub(r'[^a-zA-Z0-9]', '', company_name.lower())
            clean_domain = re.sub(r'[^a-zA-Z0-9]', '', domain.lower())
            
            if clean_name and clean_domain:
                set1 = set(clean_name)
                set2 = set(clean_domain)
                intersection = len(set1 & set2)
                union = len(set1 | set2)
                similarity = intersection / union if union > 0 else 0
            else:
                similarity = 0
            
            # Feature 3: acronym_match
            company_words = company_name.split()
            company_acronym = ''.join([word[0].lower() for word in company_words if word[0].isalpha()])
            domain_parts = domain.split('.')[0].split('-')
            domain_acronym = ''.join([part[0].lower() for part in domain_parts if part])
            acronym_match = 1 if company_acronym and domain_acronym and company_acronym == domain_acronym else 0
            
            # Feature 4: is_socialmedia
            social_platforms = ['linkedin', 'facebook', 'twitter', 'instagram', 'youtube', 'tiktok', 'snapchat']
            is_socialmedia = 1 if any(platform in domain.lower() for platform in social_platforms) else 0
            
            return [position_feature, similarity, acronym_match, is_socialmedia]
            
        except Exception as e:
            self.logger.error(f"Error preparing features: {e}")
            return None
    
    def get_google_search_results(self, query: str, num_results: int = 10) -> List[Dict]:
        """Get Google search results via Serper API"""
        try:
            params = {
                'api_key': self.serper_api_key,
                'q': query,
                'num': num_results
            }
            
            response = requests.get('https://google.serper.dev/search', params=params)
            response.raise_for_status()
            data = response.json()
            
            return data.get('organic', [])
            
        except Exception as e:
            self.logger.error(f"Error fetching search results: {e}")
            return []
    
    def _is_valid_company_domain(self, domain: str) -> bool:
        """Validate if domain is a legitimate company domain"""
        if not domain:
            return False
        
        # Blacklist
        blacklist = [
            'linkedin.com', 'facebook.com', 'twitter.com', 'instagram.com',
            'youtube.com', 'wikipedia.org', 'glassdoor.com', 'indeed.com',
            'crunchbase.com', 'bloomberg.com', 'reuters.com', 'static.licdn.com'
        ]
        
        domain_lower = domain.lower()
        
        for blocked in blacklist:
            if blocked in domain_lower:
                return False
        
        # Must have a dot
        if '.' not in domain:
            return False
        
        # Must end with valid TLD
        valid_tlds = ['.com', '.org', '.net', '.io', '.co', '.uk', '.de', '.fr', '.cn', '.jp', '.in', '.ai']
        if not any(domain_lower.endswith(tld) for tld in valid_tlds):
            return False
        
        return True
    
    def process_batch(self, company_names: List[str]) -> Dict[str, Tuple[Optional[str], str]]:
        """Process multiple company names"""
        results = {}
        total = len(company_names)
        
        print(f"\n📊 Processing {total} companies...")
        print("-" * 60)
        
        for idx, company_name in enumerate(tqdm(company_names, desc="Processing companies")):
            self.logger.info(f"Processing {idx+1}/{total}: {company_name}")
            
            print(f"\n{idx+1}/{total}: {company_name}")
            
            domain, source = self.resolve_domain(company_name)
            results[company_name] = (domain, source)
            
            status = "✅" if domain else "❌"
            print(f"   {status} Result: {domain or 'Not Found'} (Source: {source})")
            
            # Rate limiting
            time.sleep(1)
        
        return results

print("✅ Domain Resolver loaded successfully!")

✅ Domain Resolver loaded successfully!


In [9]:
# ==================== RUN THE PIPELINE ====================

print("\n" + "="*60)
print("🚀 STARTING DOMAIN RESOLUTION PIPELINE")
print("="*60)

# Initialize resolver
resolver = CompanyDomainResolver(SERPER_API_KEY, xgb_model)

# Process companies
print(f"\n📊 Processing {len(COMPANY_NAMES)} companies...")
print("-"*60)

start_time = time.time()
results = resolver.process_batch(COMPANY_NAMES)
end_time = time.time()

# Display results
print("\n" + "="*60)
print("📊 RESULTS")
print("="*60)

success_count = 0
for company, (domain, source) in results.items():
    status = "✅" if domain else "❌"
    if domain:
        success_count += 1
    print(f"{status} {company:35} → {domain or 'Not Found':30} (Source: {source})")

print("="*60)
print(f"⏱️  Time taken: {(end_time - start_time):.2f} seconds")
print(f"✅ Successfully resolved: {success_count}/{len(COMPANY_NAMES)}")
print("="*60)

# Create DataFrame with results
df_results = pd.DataFrame([
    {'Company': company, 'Domain': domain, 'Source': source}
    for company, (domain, source) in results.items()
])

# Merge with original CSV data if available
if 'df' in locals():
    try:
        # Try to merge with original data
        df_final = df.merge(df_results, left_on='company_name', right_on='Company', how='left')
        print(f"\n✅ Merged results with original CSV data")
    except:
        df_final = df_results
else:
    df_final = df_results

print("\n📊 Results DataFrame created successfully!")


🚀 STARTING DOMAIN RESOLUTION PIPELINE

📊 Processing 49 companies...
------------------------------------------------------------

📊 Processing 49 companies...
------------------------------------------------------------


Processing companies:   0%|          | 0/49 [00:00<?, ?it/s]


1/49: MakeMyTrip
   ✅ Result: careers.makemytrip.com (Source: LinkedIn)


Processing companies:   2%|▏         | 1/49 [00:15<12:36, 15.75s/it]


2/49: Wissen Infotech
   Attempt 1 failed to find website
   Waiting 5.6 seconds before retry...
   ✅ Result: wisseninfotech.com (Source: LinkedIn)


...


48/49: SRF Limited
   ✅ Result: srf.com (Source: LinkedIn)


Processing companies:  98%|█████████▊| 48/49 [27:46<00:28, 28.84s/it]


49/49: Aarti Industries
   Attempt 1 failed to find website
   Waiting 6.5 seconds before retry...
   Attempt 2 failed to find website
   Waiting 9.5 seconds before retry...
   ✅ Result: aarti-industries.com (Source: LinkedIn)


Processing companies: 100%|██████████| 49/49 [28:53<00:00, 35.37s/it]


📊 RESULTS
✅ MakeMyTrip                          → careers.makemytrip.com         (Source: LinkedIn)
✅ Wissen Infotech                     → wisseninfotech.com             (Source: LinkedIn)
✅ Khaitan & Co                        → khaitanco.com                  (Source: LinkedIn)
✅ MosChip Technologies                → moschip.com                    (Source: LinkedIn)
✅ TTK Healthcare                      → ttkhealthcare.com              (Source: LinkedIn)
❌ Jawel Circuit                       → Not Found                      (Source: Failed)
✅ Persistent Systems                  → persistentsystems.com          (Source: ML_Fallback)
✅ Zensar Technologies                 → zensar.com                     (Source: LinkedIn)
✅ Affle India                         → affle.com                      (Source: ML_Fallback)
✅ Minda Corporation                   → minda.com                      (Source: LinkedIn)
✅ Bharat Forge (mid entities)         → bharatforge.com                (Source: Linke

In [10]:
# ==================== GET CSV OUTPUT (SIMPLIFIED) ====================
import pandas as pd
from datetime import datetime

# Create results DataFrame
df_output = pd.DataFrame([
    {'Company': company, 'Domain': domain, 'Source': source}
    for company, (domain, source) in results.items()
])

# Save as CSV in current directory
csv_filename = f'company_domains_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
df_output.to_csv(csv_filename, index=False)

print(f"✅ CSV saved to: {csv_filename}")
print(f"📊 Total companies: {len(df_output)}")
print(f"✅ Successfully resolved: {len(df_output[df_output['Domain'].notna()])}")
print(f"❌ Failed: {len(df_output[df_output['Domain'].isna()])}")

# Show the first few rows
print("\n📊 Preview:")
print(df_output.head(10).to_string(index=False))

✅ CSV saved to: company_domains_20260701_140849.csv
📊 Total companies: 49
✅ Successfully resolved: 48
❌ Failed: 1

📊 Preview:
             Company                 Domain      Source
          MakeMyTrip careers.makemytrip.com    LinkedIn
     Wissen Infotech     wisseninfotech.com    LinkedIn
        Khaitan & Co          khaitanco.com    LinkedIn
MosChip Technologies            moschip.com    LinkedIn
      TTK Healthcare      ttkhealthcare.com    LinkedIn
       Jawel Circuit                    NaN      Failed
  Persistent Systems  persistentsystems.com ML_Fallback
 Zensar Technologies             zensar.com    LinkedIn
         Affle India              affle.com ML_Fallback
   Minda Corporation              minda.com    LinkedIn
